# 03: Feature Engineering, Non-Linear Transforms & Selection

**Track 02: Data Analytics, EDA & High-Performance Dataframes** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Transform raw features into predictive signals: Target Encoding with smoothing, Polynomial combinations, Box-Cox/Yeo-Johnson power transforms, Mutual Information, and RFECV selection.


## 1. Ingest Housing Prices Dataset & Engineer Ratios
Construct domain-specific interaction features: Price per sqft, bath-to-bed ratio, age.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.feature_selection import mutual_info_regression

df_housing = load_dataset("housing_prices")
print(f"Loaded Housing Dataset: {df_housing.shape}")

# Feature Engineering
df_housing["Price_Per_SqFt"] = df_housing["Price"] / (df_housing["SquareFeet"] + 1)
df_housing["Baths_Per_Bed"] = df_housing["Bathrooms"] / (df_housing["Bedrooms"] + 0.1)
df_housing["Age"] = 2026 - df_housing["YearBuilt"]

print(df_housing[["Price", "Price_Per_SqFt", "Baths_Per_Bed", "Age"]].head())

## 2. Power Transformations for Skewed Targets & Features
Applying Yeo-Johnson transformation to stabilize variance.

In [ ]:
pt = PowerTransformer(method="yeo-johnson")
df_housing["Price_Transformed"] = pt.fit_transform(df_housing[["Price"]])

print(f"Original Price Skewness    : {df_housing['Price'].skew():.4f}")
print(f"Transformed Price Skewness : {df_housing['Price_Transformed'].skew():.4f}")

## 3. Mutual Information Feature Importance
Rank all continuous and engineered features according to their non-linear mutual information with Target Price.

In [ ]:
feature_cols = ["SquareFeet", "Bedrooms", "Bathrooms", "YearBuilt", "Age", "Baths_Per_Bed"]
X = df_housing[feature_cols].fillna(0)
y = df_housing["Price"]

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.DataFrame({"Feature": feature_cols, "Mutual_Info_Score": mi_scores}).sort_values(by="Mutual_Info_Score", ascending=False)
print("=== Mutual Information Feature Ranking ===")
print(mi_df.to_string(index=False))